# Semana 7 — M3 Soluciones: Optimización y Gobernanza
**Bootcamp:** Fundamentos de Ingeniería de Datos — Databricks SQL

### Ejercicio 3.1 — GUIDED

In [0]:
%sql
-- ANTES: ejecutar y anotar el tiempo
SELECT partido, COUNT(*) as cnt, ROUND(AVG(precio), 2) as avg_precio
FROM bootcamp.semantica.v_propiedades_completa
WHERE moneda = 'USD'
GROUP BY partido
ORDER BY cnt DESC;

In [0]:
%sql
-- OPTIMIZE reorganiza small files + ZORDER coloca datos con zona_id similar en los mismos archivos
OPTIMIZE bootcamp.gold.fact_propiedades ZORDER BY (zona_id);

In [0]:
%sql
-- DESPUÉS: misma query. ZORDER permite data skipping sobre zona_id
SELECT partido, COUNT(*) as cnt, ROUND(AVG(precio), 2) as avg_precio
FROM bootcamp.semantica.v_propiedades_completa
WHERE moneda = 'USD'
GROUP BY partido
ORDER BY cnt DESC;

### Ejercicio 3.2 — INDEPENDENT

In [0]:
%sql
-- Ver versiones antes del VACUUM
DESCRIBE HISTORY bootcamp.gold.fact_propiedades;

In [0]:
%sql
-- VACUUM elimina archivos que ya no son referenciados por versiones dentro del período de retención
VACUUM bootcamp.gold.fact_propiedades RETAIN 168 HOURS;

In [0]:
%sql
-- Respuestas:
-- 1. Depende de cuántas operaciones se hicieron (ver output de DESCRIBE HISTORY)
-- 2. VACUUM con 0 horas elimina TODOS los archivos de versiones anteriores.
--    Time Travel deja de funcionar para cualquier versión previa.
--    Requiere desactivar el safety check (muy peligroso en producción).
-- 3. Podés seguir usando Time Travel para versiones dentro del período de retención.
--    Las versiones más viejas que 168 horas pierden sus archivos.
DESCRIBE HISTORY bootcamp.gold.fact_propiedades;

### Ejercicio 3.3 — INDEPENDENT

In [0]:
 %sql
SHOW GRANTS ON TABLE bootcamp.gold.fact_propiedades;

-- Comandos de gobernanza (ejecutar si tenés permisos de admin):
-- 1. GRANT SELECT ON VIEW bootcamp.semantica.v_propiedades_completa TO `analistas`;
-- 2. REVOKE MODIFY ON TABLE bootcamp.gold.fact_propiedades FROM `analistas`;
-- 3. GRANT SELECT ON SCHEMA bootcamp.semantica TO `analistas`;
--    (GRANT USAGE no disponible en privilege version 1.0 del metastore)

-- ¿Por qué dar acceso a la view pero no a la tabla?
-- La view es la interfaz controlada: muestra solo lo que el consumidor necesita.
-- Si le das acceso directo a la tabla, puede ver columnas sensibles,
-- hacer JOINs incorrectos, o modificar datos y romper las optimizaciones.

---
## Reflexión — Respuestas sugeridas

1. **APIs para DEs:** Un Data Engineer necesita ingestar datos de fuentes externas constantemente — cotizaciones, clima, redes sociales, APIs internas de la empresa. Sin saber trabajar con APIs, dependés de que alguien más te dé los datos en CSV. En la realidad, la mayoría de las fuentes de datos modernas exponen APIs REST.

2. **View vs Materialized View:** Una View es una query guardada — se ejecuta cada vez que la consultás (siempre datos frescos, pero más lenta). Una Materialized View pre-computa y almacena el resultado (más rápida, pero puede tener datos stale). Usás MV cuando la query es pesada y los datos no cambian cada minuto.

3. **Gobernanza protege optimizaciones:** Si alguien hace un INSERT INTO sin criterio en una tabla optimizada con ZORDER, los datos nuevos quedan desordenados y rompen el data skipping. Si cambian el schema sin avisar, los pipelines downstream se rompen. Gobernanza (GRANT/REVOKE) asegura que solo las personas correctas puedan modificar las tablas.

4. **Capa semántica en 2 oraciones:** La capa semántica son views que simplifican las tablas complejas del data warehouse para que el negocio pueda consultar datos sin necesitar saber SQL avanzado ni entender el modelo dimensional. Es la interfaz entre el Data Engineer y el consumidor final.